# 階段 5：第二層 —— RSI 回檔進場訊號

台股擇時策略研究專案第五階段。

| 組別 | 第一層濾網 | RSI 擇時 | 狀態 |
|---|---|---|---|
| A 買進持有 | ✗ | ✗ | 階段 4 完成 |
| C 只有濾網 | ✓ | ✗ | 階段 4 完成 |
| **B 完整策略** | ✓ | **✓** | 階段 5、6 |

**B 與 C 唯一的差別是進場時點。** 濾網、出場規則、曝險方式完全相同。
C 組在狀態轉多當日開盤即進場；B 組要等 RSI 給出回檔結束的訊號才進場。

因此 `B − C` 的差異，就是 RSI 這一層的貢獻。
本階段先把訊號本身看清楚，**不做回測、不計算任何績效數字**，階段 6 才做回測。

### 沿用階段 1–4 的結論（已確認，不重複檢查）

- `data/regime.csv`，6,632 筆，1999-01-05 ~ 2026-01-20
- RSI 為 Wilder 平滑，已用純迴圈交叉驗證
- `regime` 已驗證無 look-ahead，2000-01-01 之後無缺值
- 研究期間 24 次 `FLAT → LONG_OK`，加上期初已在場的一段，C 組共 25 筆交易
- 研究期間 RSI3 落在 30 以下者佔 23.50%

**產出**：`data/signals.csv`、`data/signal_comparison.csv`

---
## 規則定義

### 第二層進場邏輯（狀態機，逐日推進）

```
在 regime == LONG_OK 且尚未持有部位時：
    若 RSI3[t] < 30            → 設定 pullback_flag = True
    若 pullback_flag == True
       且 RSI3[t] 上穿 RSI5[t]  → 於 t+1 開盤進場，清除 pullback_flag

在 regime 由 FLAT 轉為 LONG_OK 的那一天：
    清空 pullback_flag（新的多頭段落重新計算）

已持有部位時：忽略所有新訊號，不加碼
```

### 關鍵細節

1. **「RSI3 < 30」是當日收盤值落在 30 以下即可**，不是「穿越 30」。
2. **「上穿」的定義**：`RSI3[t] > RSI5[t]` 且 `RSI3[t−1] <= RSI5[t−1]`。
3. **訊號在 t 日收盤確認，t+1 日開盤成交。**
4. **`pullback_flag` 只在 `LONG_OK` 狀態下累積**，`FLAT` 期間即使 RSI3 跌破 30 也不記錄。
5. 若訊號的 t+1 日已變成 `FLAT`（狀態剛好在月初切換），**該訊號作廢，不進場**。

### ★ 兩層的時序不同，容易搞混

| | 判斷時點 | 生效／成交時點 | 本階段是否要 shift |
|---|---|---|---|
| 第一層 `regime` | 月底收盤 | 次月第一個交易日開盤 | **不用** —— 階段 3 已處理，`regime` 欄位就是生效後的序列 |
| 第二層 RSI 訊號 | t 日收盤 | **t+1 日開盤** | **要** —— 訊號日 +1 個交易日才是進場日 |

第一層再 shift 一次會讓進場延後一天；第二層忘了 shift 則會用收盤後才知道的資訊在當天成交。
兩種錯誤方向相反，區塊 4(d) 會明確確認。

### 期初處理

研究期間第一天 2000-01-04 的 `regime` 已是 `LONG_OK`（承接 1999-12 月底的判斷）。
這一段視為一個正常的 `LONG_OK` 區間，`pullback_flag` 從該日起算為 `False`，
要等 RSI3 跌破 30 才會啟動。

**C 組在這一段是期初就進場，B 組則要等訊號** —— 這個差異會在區塊 5 明確呈現。

---
## 1. 參數與載入

參數集中定義。載入後比對筆數、起訖日期與欄位是否與階段 3 一致，不一致就 `raise`。

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============ 參數 ============
DATA_IN      = "data/regime.csv"
DATA_OUT     = "data/signals.csv"
COMPARE_OUT  = "data/signal_comparison.csv"
STUDY_START  = "2000-01-01"
IS_END       = "2017-12-31"
OOS_START    = "2018-01-01"
RSI_OVERSOLD = 30

LONG_OK, FLAT = "LONG_OK", "FLAT"

# ============ 階段 3、4 已確認的事實 ============
EXPECTED_ROWS  = 6632
EXPECTED_FIRST = "1999-01-05"
EXPECTED_LAST  = "2026-01-20"
EXPECTED_COLS  = ["Open", "High", "Low", "Close", "Volume", "MA",
                  "RSI3", "RSI5", "regime", "is_month_end", "regime_change"]
EXPECTED_BLOCKS = 50       # 研究期間 LONG_OK 區間數（= C 組交易筆數）

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 400)
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (14, 7)

# ============ 驗證結果收集器（沿用前面階段模式） ============
CHECKS = []

def record(name, passed, detail=""):
    verdict = "通過" if passed else "異常"
    CHECKS.append({"驗證項目": name, "結果": verdict, "說明": detail})
    print(f"===> [{verdict}] {name}" + (f"\n      {detail}" if detail else ""))

print("參數設定完成")
print(f"  研究期間      : {STUDY_START} ~ {EXPECTED_LAST}")
print(f"  RSI 超賣門檻  : RSI3 < {RSI_OVERSOLD}")
print(f"  上穿定義      : RSI3[t] > RSI5[t] 且 RSI3[t-1] <= RSI5[t-1]")

參數設定完成
  研究期間      : 2000-01-01 ~ 2026-01-20
  RSI 超賣門檻  : RSI3 < 30
  上穿定義      : RSI3[t] > RSI5[t] 且 RSI3[t-1] <= RSI5[t-1]


In [2]:
raw = pd.read_csv(DATA_IN, index_col="Date", parse_dates=True).sort_index()

load_check = pd.DataFrame({
    "階段 3 確認值": [str(EXPECTED_ROWS), EXPECTED_FIRST, EXPECTED_LAST,
                      ", ".join(EXPECTED_COLS)],
    "本次載入值":    [f"{len(raw)}", f"{raw.index.min():%Y-%m-%d}",
                      f"{raw.index.max():%Y-%m-%d}", ", ".join(raw.columns)],
}, index=["筆數", "起始日期", "結束日期", "欄位"])
load_check["一致"] = np.where(
    load_check["階段 3 確認值"] == load_check["本次載入值"], "是", "★ 否")
display(load_check)

if (load_check["一致"] == "★ 否").any():
    raise RuntimeError("載入的資料與階段 3 不一致，請停止並人工確認 data/regime.csv")

df = raw.loc[STUDY_START:].copy()

record("資料載入一致性", True,
       f"{len(raw):,} 筆與階段 3 完全一致；研究期間切出 {len(df):,} 個交易日"
       f"（{df.index.min():%Y-%m-%d} ~ {df.index.max():%Y-%m-%d}）")

,階段 3 確認值,本次載入值,一致
筆數,6632,6632,是
起始日期,1999-01-05,1999-01-05,是
結束日期,2026-01-20,2026-01-20,是
欄位,"Open, High, Low, Close, Volume, MA, RSI3, RSI5...","Open, High, Low, Close, Volume, MA, RSI3, RSI5...",是


===> [通過] 資料載入一致性
      6,632 筆與階段 3 完全一致；研究期間切出 6,391 個交易日（2000-01-04 ~ 2026-01-20）


---
## 2. 切出 LONG_OK 區間

把研究期間切成連續的 `LONG_OK` 區間，每段給一個編號。**這是本階段的分析單位** ——
每個區間對應 C 組的一筆交易，B 組在同一個區間內最多也只會進場一次。

區間數量應與階段 4 的 C 組交易筆數一致（25 筆，其中最後一筆在資料結束時仍持續）。

In [3]:
blk_id = (df["regime"] != df["regime"].shift(1)).cumsum()
df["regime_block"] = np.where(df["regime"] == LONG_OK, blk_id, np.nan)

# 重新編號成 1, 2, 3, ...
codes = {v: i + 1 for i, v in enumerate(sorted(df["regime_block"].dropna().unique()))}
df["regime_block"] = df["regime_block"].map(codes)

rows = []
for b, seg in df.dropna(subset=["regime_block"]).groupby("regime_block"):
    rows.append({
        "區間#": int(b),
        "起始日": seg.index[0], "結束日": seg.index[-1],
        "交易日數": len(seg),
        "起始日開盤": seg["Open"].iloc[0],
        "結束日開盤": seg["Open"].iloc[-1],
    })
blocks = pd.DataFrame(rows).set_index("區間#")

# 最後一個區間是否延續到資料結束
blocks["資料結束時仍持續"] = blocks["結束日"] == df.index[-1]

blocks_disp = blocks.copy()
for c in ["起始日", "結束日"]:
    blocks_disp[c] = blocks_disp[c].dt.strftime("%Y-%m-%d")
display(blocks_disp.style.format({"起始日開盤": "{:,.2f}", "結束日開盤": "{:,.2f}"}))

,起始日,結束日,交易日數,起始日開盤,結束日開盤,資料結束時仍持續
區間#,,,,,,
1,2000-01-04,2000-04-28,76,"8,644.91","8,594.48",False
2,2001-02-01,2001-04-30,61,"5,927.25","5,430.59",False
3,2001-12-03,2002-04-30,97,"4,534.37","6,201.84",False
4,2002-11-01,2002-12-31,43,"4,596.69","4,455.92",False
5,2003-02-06,2003-02-27,16,"4,975.65","4,411.54",False
6,2003-06-02,2003-11-28,127,"4,620.54","5,804.82",False
7,2004-02-02,2004-03-31,43,"6,379.98","6,554.95",False
8,2004-09-01,2004-10-29,41,"5,799.82","5,693.11",False
9,2005-01-03,2005-03-31,57,"6,166.39","6,005.87",False


In [4]:
n_blocks = len(blocks)
blocks_ok = n_blocks == EXPECTED_BLOCKS

record(f"{LONG_OK} 區間切分", blocks_ok,
       f"研究期間切出 {n_blocks} 個 {LONG_OK} 區間，與階段 4 的 C 組交易筆數 "
       f"{EXPECTED_BLOCKS} 一致；最後一段（區間 {n_blocks}，"
       f"{blocks['起始日'].iloc[-1]:%Y-%m-%d} 起）於資料結束時仍持續"
       if blocks_ok else f"區間數 {n_blocks} 與預期的 {EXPECTED_BLOCKS} 不符")

===> [通過] LONG_OK 區間切分
      研究期間切出 50 個 LONG_OK 區間，與階段 4 的 C 組交易筆數 50 一致；最後一段（區間 50，2025-06-02 起）於資料結束時仍持續


---
## 3. 實作狀態機

**用明確的逐日迴圈實作，不向量化。**

這一層有狀態依賴（`pullback_flag`、是否已持有），向量化寫法容易錯且難以檢視。
這裡可讀性優先 —— 迴圈的每一行都能直接對應到規則的一句話。

同一天內的處理順序照規則書寫的順序：先看 `RSI3 < 30` 設旗標，再看是否上穿。
因此「當日 RSI3 跌破 30 且同日上穿 RSI5」會在當日就成立訊號（實務上罕見，但規則如此）。

In [5]:
idx    = df.index
regime = df["regime"].to_numpy()
rsi3   = df["RSI3"].to_numpy()
rsi5   = df["RSI5"].to_numpy()
n      = len(df)

pullback_flag = np.zeros(n, dtype=bool)
entry_signal  = np.zeros(n, dtype=bool)
signal_void   = np.zeros(n, dtype=bool)   # 訊號成立但 t+1 已轉 FLAT，作廢

flag   = False
in_pos = False
prev_regime = None

for t in range(n):
    reg = regime[t]

    # --- 狀態轉換處理 ---
    if reg == LONG_OK and prev_regime != LONG_OK:
        flag = False            # 新的多頭段落，旗標重新計算
    if reg == FLAT:
        in_pos = False          # 第一層已出場
        flag = False            # FLAT 期間不累積旗標

    # --- 訊號邏輯：僅在 LONG_OK 且尚未持有時 ---
    if reg == LONG_OK and not in_pos:
        if rsi3[t] < RSI_OVERSOLD:
            flag = True

        crossed_up = (t > 0) and (rsi3[t] > rsi5[t]) and (rsi3[t - 1] <= rsi5[t - 1])
        if flag and crossed_up:
            entry_signal[t] = True
            flag = False
            # t+1 開盤成交；若 t+1 已轉 FLAT 或無 t+1，訊號作廢
            if t + 1 < n and regime[t + 1] == LONG_OK:
                in_pos = True
            else:
                signal_void[t] = True

    pullback_flag[t] = flag
    prev_regime = reg

df["pullback_flag"] = pullback_flag
df["entry_signal"]  = entry_signal
df["signal_void"]   = signal_void

# B 組進場日 = 訊號日 + 1 個交易日（作廢者除外）
b_entry = np.zeros(n, dtype=bool)
for t in np.where(entry_signal & ~signal_void)[0]:
    b_entry[t + 1] = True
df["b_entry_day"] = b_entry

summary0 = pd.DataFrame({"值": [
    int(entry_signal.sum()), int(signal_void.sum()), int(b_entry.sum()),
    int(pullback_flag.sum()), f"{pullback_flag.sum() / n:.2%}",
]}, index=["entry_signal 成立天數", "  └ 其中作廢", "b_entry_day 天數",
           "pullback_flag == True 天數", "  └ 佔研究期間"])
summary0.index.name = "狀態機輸出"
display(summary0)

,值
狀態機輸出,
entry_signal 成立天數,50
└ 其中作廢,1
b_entry_day 天數,49
pullback_flag == True 天數,161
└ 佔研究期間,2.52%


---
## 4. 驗證狀態機 ★ 本階段核心驗證

四項驗證：(a) 旗標邏輯、(b) 訊號條件、(c) 每區間最多一次、(d) 時序方向。

### (a) 旗標邏輯

兩件事要成立：

1. 每個 `pullback_flag == True` 的日子，往前在**同一區間內**必定找得到一個 `RSI3 < 30`，
   且從那天到今天之間沒有進場訊號（有訊號就該被清除）
2. `FLAT` 期間 `pullback_flag` 必須全為 `False`

這裡刻意不重跑一次同樣的迴圈 —— 那只會把同樣的錯誤重現。
改成從結果反推條件是否成立。

In [6]:
flag_bad = []
for t in np.where(pullback_flag)[0]:
    b = df["regime_block"].iloc[t]
    if pd.isna(b):
        flag_bad.append((t, "FLAT 期間旗標為 True"))
        continue
    # 同一區間內、t 當日或之前，最後一個 RSI3 < 30 的位置
    seg_pos = np.where((df["regime_block"].to_numpy() == b) &
                       (np.arange(n) <= t) & (rsi3 < RSI_OVERSOLD))[0]
    if len(seg_pos) == 0:
        flag_bad.append((t, "區間內找不到觸發旗標的 RSI3 < 30"))
        continue
    s = seg_pos[-1]
    if entry_signal[s:t].any():          # [s, t-1] 之間不得有訊號
        flag_bad.append((t, "觸發日之後、當日之前出現過訊號，旗標應已清除"))

flat_flag = int(pullback_flag[df["regime"].to_numpy() == FLAT].sum())

tbl = pd.DataFrame({
    "檢查": ["旗標日皆可回溯到區間內的 RSI3 < 30 且中間無訊號",
             "FLAT 期間 pullback_flag 全為 False"],
    "違反天數": [len(flag_bad), flat_flag],
})
tbl["結果"] = np.where(tbl["違反天數"] == 0, "通過", "★ 異常")
display(tbl)
if flag_bad:
    display(pd.DataFrame([{"日期": f"{idx[t]:%Y-%m-%d}", "原因": r} for t, r in flag_bad[:20]]))

a_ok = (tbl["違反天數"] == 0).all()
record("(a) 旗標邏輯", a_ok,
       f"{int(pullback_flag.sum()):,} 個 pullback_flag 為 True 的交易日全部可回溯到"
       f"同一區間內的 RSI3 < {RSI_OVERSOLD}，且中間無未清除的訊號；"
       f"FLAT 期間旗標全為 False"
       if a_ok else "旗標邏輯有問題，明細見上表")

,檢查,違反天數,結果
0,旗標日皆可回溯到區間內的 RSI3 < 30 且中間無訊號,0,通過
1,FLAT 期間 pullback_flag 全為 False,0,通過


===> [通過] (a) 旗標邏輯
      161 個 pullback_flag 為 True 的交易日全部可回溯到同一區間內的 RSI3 < 30，且中間無未清除的訊號；FLAT 期間旗標全為 False


### (b) 訊號條件

逐一檢查每個 `entry_signal == True` 的日子，確認同時滿足四個條件。
寫成程式化斷言，任何一筆不符就算異常。

In [7]:
sig_rows = []
for t in np.where(entry_signal)[0]:
    b = df["regime_block"].iloc[t]
    seg_mask = (df["regime_block"].to_numpy() == b) & (np.arange(n) <= t)
    prev_entries = entry_signal[:t][ (df["regime_block"].to_numpy()[:t] == b) ] if not pd.isna(b) else []
    sig_rows.append({
        "訊號日": f"{idx[t]:%Y-%m-%d}",
        "區間#": "—" if pd.isna(b) else int(b),
        "regime == LONG_OK": regime[t] == LONG_OK,
        "RSI3 > RSI5": bool(rsi3[t] > rsi5[t]),
        "前一日 RSI3 <= RSI5": bool(rsi3[t - 1] <= rsi5[t - 1]),
        f"區間內先前出現 RSI3 < {RSI_OVERSOLD}": bool((rsi3[seg_mask] < RSI_OVERSOLD).any()),
        "區間內此前未進場": bool(np.asarray(prev_entries).sum() == 0),
    })
sig_chk = pd.DataFrame(sig_rows)
COND = [c for c in sig_chk.columns if c not in ("訊號日", "區間#")]
sig_chk["全部滿足"] = sig_chk[COND].all(axis=1)
display(sig_chk)

b_ok = bool(sig_chk["全部滿足"].all())
record("(b) 訊號條件", b_ok,
       f"{len(sig_chk)} 個 entry_signal 全部同時滿足四個條件："
       f"regime 為 {LONG_OK}、當日 RSI3 > RSI5、前一日 RSI3 <= RSI5、"
       f"區間內先前出現過 RSI3 < {RSI_OVERSOLD}、且區間內此前未進場"
       if b_ok else "有訊號不滿足條件，明細見上表")

,訊號日,區間#,regime == LONG_OK,RSI3 > RSI5,前一日 RSI3 <= RSI5,區間內先前出現 RSI3 < 30,區間內此前未進場,全部滿足
0,2000-03-01,1,True,True,True,True,True,True
1,2001-02-12,2,True,True,True,True,True,True
2,2001-12-25,3,True,True,True,True,True,True
3,2002-11-22,4,True,True,True,True,True,True
4,2003-02-17,5,True,True,True,True,True,True
5,2003-07-01,6,True,True,True,True,True,True
6,2004-02-06,7,True,True,True,True,True,True
7,2004-09-21,8,True,True,True,True,True,True
8,2005-01-11,9,True,True,True,True,True,True
9,2005-07-01,10,True,True,True,True,True,True


===> [通過] (b) 訊號條件
      50 個 entry_signal 全部同時滿足四個條件：regime 為 LONG_OK、當日 RSI3 > RSI5、前一日 RSI3 <= RSI5、區間內先前出現過 RSI3 < 30、且區間內此前未進場


### (c) 每區間最多一次

In [8]:
per_block = df.dropna(subset=["regime_block"]).groupby("regime_block")["entry_signal"].sum()
over = per_block[per_block > 1]

dist = per_block.value_counts().sort_index().to_frame("區間數")
dist.index.name = "區間內訊號次數"
display(dist)

c_ok = len(over) == 0
record("(c) 每區間最多一次訊號", c_ok,
       f"{len(per_block)} 個 {LONG_OK} 區間中，"
       f"{int((per_block == 1).sum())} 個產生 1 次訊號、"
       f"{int((per_block == 0).sum())} 個未產生訊號，無任何區間超過 1 次"
       if c_ok else f"有 {len(over)} 個區間訊號超過 1 次")

,區間數
區間內訊號次數,
1,50


===> [通過] (c) 每區間最多一次訊號
      50 個 LONG_OK 區間中，50 個產生 1 次訊號、0 個未產生訊號，無任何區間超過 1 次


### (d) 時序確認 —— shift 方向正確

印出前 5 筆訊號的前後各 2 天。要確認的是：

- `entry_signal == True` 在 **t 日**（該日收盤才知道 RSI3 是否上穿）
- `b_entry_day == True` 在 **t+1 日**，成交價用該日 `Open`

若兩者出現在同一天，代表忘了 shift（用收盤後的資訊在當天開盤成交，是 look-ahead）；
若相差兩天以上，代表 shift 過頭。

In [9]:
sig_pos = np.where(entry_signal & ~signal_void)[0]

for t in sig_pos[:5]:
    win = df.iloc[max(t - 2, 0): t + 3][
        ["Open", "Close", "RSI3", "RSI5", "regime",
         "pullback_flag", "entry_signal", "b_entry_day"]].copy()
    win["◀"] = np.where(win.index == idx[t], "◀ 訊號日",
                np.where(win.index == idx[t + 1], "◀ 進場日（開盤成交）", ""))
    win.index = win.index.strftime("%Y-%m-%d")
    print(f"\n=== 訊號 {idx[t]:%Y-%m-%d} → 進場 {idx[t + 1]:%Y-%m-%d} ===")
    display(win.style.format({"Open": "{:,.2f}", "Close": "{:,.2f}",
                              "RSI3": "{:.2f}", "RSI5": "{:.2f}"}))


=== 訊號 2000-03-01 → 進場 2000-03-02 ===


,Open,Close,RSI3,RSI5,regime,pullback_flag,entry_signal,b_entry_day,◀
Date,,,,,,,,,
2000-02-25,"9,620.92","9,432.49",6.17,17.54,LONG_OK,True,False,False,
2000-02-29,"9,525.65","9,435.94",7.48,18.14,LONG_OK,True,False,False,
2000-03-01,"9,572.24","9,689.10",63.48,50.84,LONG_OK,False,True,False,◀ 訊號日
2000-03-02,"9,781.27","9,543.82",41.73,39.52,LONG_OK,False,False,True,◀ 進場日（開盤成交）
2000-03-03,"9,557.66","9,588.03",49.61,44.24,LONG_OK,False,False,False,



=== 訊號 2001-02-12 → 進場 2001-02-13 ===


,Open,Close,RSI3,RSI5,regime,pullback_flag,entry_signal,b_entry_day,◀
Date,,,,,,,,,
2001-02-08,"5,693.63","5,758.60",39.48,46.87,LONG_OK,True,False,False,
2001-02-09,"5,782.42","5,809.84",51.51,52.49,LONG_OK,True,False,False,
2001-02-12,"5,807.61","5,847.07",60.15,56.65,LONG_OK,False,True,False,◀ 訊號日
2001-02-13,"5,922.02","6,027.49",82.63,71.68,LONG_OK,False,False,True,◀ 進場日（開盤成交）
2001-02-14,"6,060.89","5,887.68",49.90,53.66,LONG_OK,False,False,False,



=== 訊號 2001-12-25 → 進場 2001-12-26 ===


,Open,Close,RSI3,RSI5,regime,pullback_flag,entry_signal,b_entry_day,◀
Date,,,,,,,,,
2001-12-21,"5,209.27","5,109.24",23.20,35.76,LONG_OK,True,False,False,
2001-12-24,"5,132.61","5,164.73",36.50,42.26,LONG_OK,True,False,False,
2001-12-25,"5,198.17","5,372.81",67.84,60.83,LONG_OK,False,True,False,◀ 訊號日
2001-12-26,"5,421.73","5,392.43",69.93,62.26,LONG_OK,False,False,True,◀ 進場日（開盤成交）
2001-12-27,"5,464.52","5,332.98",53.94,54.69,LONG_OK,False,False,False,



=== 訊號 2002-11-22 → 進場 2002-11-25 ===


,Open,Close,RSI3,RSI5,regime,pullback_flag,entry_signal,b_entry_day,◀
Date,,,,,,,,,
2002-11-20,"4,721.90","4,653.50",27.34,40.25,LONG_OK,True,False,False,
2002-11-21,"4,722.06","4,579.45",17.21,30.97,LONG_OK,True,False,False,
2002-11-22,"4,694.52","4,707.61",57.80,53.95,LONG_OK,False,True,False,◀ 訊號日
2002-11-25,"4,731.31","4,723.16",61.25,56.16,LONG_OK,False,False,True,◀ 進場日（開盤成交）
2002-11-26,"4,722.35","4,677.89",45.12,47.80,LONG_OK,False,False,False,



=== 訊號 2003-02-17 → 進場 2003-02-18 ===


,Open,Close,RSI3,RSI5,regime,pullback_flag,entry_signal,b_entry_day,◀
Date,,,,,,,,,
2003-02-13,"4,616.62","4,507.96",5.25,12.39,LONG_OK,True,False,False,
2003-02-14,"4,562.24","4,493.99",4.78,11.80,LONG_OK,True,False,False,
2003-02-17,"4,629.57","4,705.08",68.30,53.47,LONG_OK,False,True,False,◀ 訊號日
2003-02-18,"4,698.10","4,605.31",46.37,41.80,LONG_OK,False,False,True,◀ 進場日（開盤成交）
2003-02-19,"4,670.15","4,550.83",36.72,36.38,LONG_OK,False,False,False,


In [10]:
# 程式化確認：每個有效訊號的進場日恰好是下一個交易日
gaps = [df.index.get_loc(idx[t + 1]) - t for t in sig_pos]
gap_ok = all(g == 1 for g in gaps)

# 且進場日的 regime 必為 LONG_OK
entry_regime_ok = bool((df.loc[df["b_entry_day"], "regime"] == LONG_OK).all())
# 訊號日與進場日不得為同一天
no_same_day = bool(not (df["entry_signal"] & df["b_entry_day"]).any())

tbl = pd.DataFrame({
    "檢查": ["進場日 = 訊號日 + 1 個交易日", "進場日 regime 為 LONG_OK",
             "訊號日與進場日不同天"],
    "結果": ["通過" if gap_ok else "★ 異常",
             "通過" if entry_regime_ok else "★ 異常",
             "通過" if no_same_day else "★ 異常"],
})
display(tbl)

d_ok = gap_ok and entry_regime_ok and no_same_day
record("(d) 時序：訊號日 t，進場日 t+1", d_ok,
       f"{len(sig_pos)} 個有效訊號的進場日皆為訊號日的下一個交易日（間隔恰為 1），"
       f"進場日 regime 皆為 {LONG_OK}，且訊號日與進場日不重疊。"
       f"第一層 regime 未再 shift、第二層訊號已 shift 一天，兩層時序正確"
       if d_ok else "時序有問題，明細見上表")

,檢查,結果
0,進場日 = 訊號日 + 1 個交易日,通過
1,進場日 regime 為 LONG_OK,通過
2,訊號日與進場日不同天,通過


===> [通過] (d) 時序：訊號日 t，進場日 t+1
      49 個有效訊號的進場日皆為訊號日的下一個交易日（間隔恰為 1），進場日 regime 皆為 LONG_OK，且訊號日與進場日不重疊。第一層 regime 未再 shift、第二層訊號已 shift 一天，兩層時序正確


---
## 5. 訊號清單 ★ 本階段核心產出

以 `LONG_OK` 區間為單位，把 C 組與 B 組的進場時點並列。

- **C 組進場日** = 區間第一天，開盤成交
- **B 組進場日** = 訊號日 + 1，開盤成交
- **等待交易日數** = B 組比 C 組晚進場幾個交易日
- **進場價差異%** = `B 組進場價 / C 組進場價 − 1`，**負值代表 B 組買得更便宜**

規格中的「錯過的漲跌%」與「進場價差異%」是同一個數字（同一個比值的兩種說法），
因此只保留一欄，避免同一資訊出現兩次造成誤讀。

未產生訊號的區間，後續欄位留空並標記。完整列出不截斷。

In [11]:
rows = []
for b, seg in df.dropna(subset=["regime_block"]).groupby("regime_block"):
    b = int(b)
    c_day = seg.index[0]
    c_px  = seg["Open"].iloc[0]

    sig = seg.index[seg["entry_signal"]]
    voided = bool(seg["signal_void"].any())
    b_days = seg.index[seg["b_entry_day"]]

    if len(b_days):
        b_day = b_days[0]
        b_px  = df.loc[b_day, "Open"]
        wait  = df.index.get_loc(b_day) - df.index.get_loc(c_day)
        diff  = b_px / c_px - 1
        status = "有訊號"
    else:
        b_day = b_px = wait = diff = np.nan
        status = "★ 訊號作廢" if voided else "★ 無訊號"

    rows.append({
        "區間#": b,
        "區間起日": f"{seg.index[0]:%Y-%m-%d}",
        "區間結束日": f"{seg.index[-1]:%Y-%m-%d}",
        "區間長度": len(seg),
        "C組進場日": f"{c_day:%Y-%m-%d}",
        "C組進場價": c_px,
        "是否產生訊號": status,
        "訊號日": f"{sig[0]:%Y-%m-%d}" if len(sig) else "—",
        "B組進場日": f"{b_day:%Y-%m-%d}" if len(b_days) else "—",
        "B組進場價": b_px,
        "等待交易日數": wait,
        "進場價差異%": diff,
    })

compare = pd.DataFrame(rows).set_index("區間#")
display(compare.style.format({
    "C組進場價": "{:,.2f}", "B組進場價": "{:,.2f}",
    "等待交易日數": "{:.0f}", "進場價差異%": "{:+.2%}"}, na_rep="—"))

,區間起日,區間結束日,區間長度,C組進場日,C組進場價,是否產生訊號,訊號日,B組進場日,B組進場價,等待交易日數,進場價差異%
區間#,,,,,,,,,,,
1,2000-01-04,2000-04-28,76,2000-01-04,"8,644.91",有訊號,2000-03-01,2000-03-02,"9,781.27",36,+13.14%
2,2001-02-01,2001-04-30,61,2001-02-01,"5,927.25",有訊號,2001-02-12,2001-02-13,"5,922.02",8,-0.09%
3,2001-12-03,2002-04-30,97,2001-12-03,"4,534.37",有訊號,2001-12-25,2001-12-26,"5,421.73",17,+19.57%
4,2002-11-01,2002-12-31,43,2002-11-01,"4,596.69",有訊號,2002-11-22,2002-11-25,"4,731.31",16,+2.93%
5,2003-02-06,2003-02-27,16,2003-02-06,"4,975.65",有訊號,2003-02-17,2003-02-18,"4,698.10",8,-5.58%
6,2003-06-02,2003-11-28,127,2003-06-02,"4,620.54",有訊號,2003-07-01,2003-07-02,"5,075.21",21,+9.84%
7,2004-02-02,2004-03-31,43,2004-02-02,"6,379.98",有訊號,2004-02-06,2004-02-09,"6,442.84",5,+0.99%
8,2004-09-01,2004-10-29,41,2004-09-01,"5,799.82",有訊號,2004-09-21,2004-09-22,"5,961.91",15,+2.79%
9,2005-01-03,2005-03-31,57,2005-01-03,"6,166.39",有訊號,2005-01-11,2005-01-12,"5,957.90",7,-3.38%


In [12]:
has_sig = compare["是否產生訊號"] == "有訊號"

record("訊號清單", True,
       f"{len(compare)} 個 {LONG_OK} 區間中，{int(has_sig.sum())} 個產生有效訊號、"
       f"{int((~has_sig).sum())} 個未產生（含訊號作廢者 "
       f"{int((compare['是否產生訊號'] == '★ 訊號作廢').sum())} 個）；"
       f"等待交易日數中位數 {compare.loc[has_sig, '等待交易日數'].median():.0f}，"
       f"進場價差異中位數 {compare.loc[has_sig, '進場價差異%'].median():+.2%}"
       "（僅記錄，不作判斷）")

===> [通過] 訊號清單
      50 個 LONG_OK 區間中，49 個產生有效訊號、1 個未產生（含訊號作廢者 1 個）；等待交易日數中位數 12，進場價差異中位數 +1.38%（僅記錄，不作判斷）


---
## 6. 訊號統計

區間數、等待天數分布、進場價差異分布，全期間與 IS / OOS 分別呈現。

區間歸屬 IS 或 OOS 依**區間起始日**判定。

**只呈現數字，不下判斷、不評價 RSI 是否有效。**

In [13]:
start_dt = pd.to_datetime(compare["區間起日"])
WINDOWS = [
    ("全期間", compare.index),
    ("IS",  compare.index[start_dt <= IS_END]),
    ("OOS", compare.index[start_dt >= OOS_START]),
]

def block_stats(sub):
    hs = sub["是否產生訊號"] == "有訊號"
    g  = sub[hs]
    no = sub[~hs]
    return pd.Series({
        f"{LONG_OK} 區間總數": len(sub),
        "有訊號區間數": int(hs.sum()),
        "無訊號區間數": int((~hs).sum()),
        "  └ 其中訊號作廢": int((sub["是否產生訊號"] == "★ 訊號作廢").sum()),
        "有訊號比例": hs.mean(),
        "無訊號區間長度中位數": no["區間長度"].median() if len(no) else np.nan,
        "無訊號區間長度最長": no["區間長度"].max() if len(no) else np.nan,
        "有訊號區間長度中位數": g["區間長度"].median() if len(g) else np.nan,
        "等待日數 最短": g["等待交易日數"].min(),
        "等待日數 中位數": g["等待交易日數"].median(),
        "等待日數 平均": g["等待交易日數"].mean(),
        "等待日數 最長": g["等待交易日數"].max(),
        "價差 中位數": g["進場價差異%"].median(),
        "價差 平均": g["進場價差異%"].mean(),
        "價差 最好（最低價買進）": g["進場價差異%"].min(),
        "價差 最差": g["進場價差異%"].max(),
        "B 組買得更便宜的比例": (g["進場價差異%"] < 0).mean() if len(g) else np.nan,
    })

stats = pd.concat([block_stats(compare.loc[ix]).rename(w) for w, ix in WINDOWS], axis=1)

SIGNED_PCT = ["價差 中位數", "價差 平均", "價差 最好（最低價買進）", "價差 最差"]
RATIO_PCT  = ["有訊號比例", "B 組買得更便宜的比例"]

stats_disp = stats.astype(object)     # 允許整列換成格式化字串
for r in stats_disp.index:
    if r in SIGNED_PCT:               # 有方向性，保留正負號
        stats_disp.loc[r] = [f"{v:+.2%}" if pd.notna(v) else "—" for v in stats.loc[r]]
    elif r in RATIO_PCT:              # 佔比，不加正負號
        stats_disp.loc[r] = [f"{v:.2%}" if pd.notna(v) else "—" for v in stats.loc[r]]
    else:
        stats_disp.loc[r] = [f"{v:,.1f}" if pd.notna(v) else "—" for v in stats.loc[r]]
display(stats_disp)

,全期間,IS,OOS
LONG_OK 區間總數,50.0,32.0,18.0
有訊號區間數,49.0,32.0,17.0
無訊號區間數,1.0,0.0,1.0
└ 其中訊號作廢,1.0,0.0,1.0
有訊號比例,98.00%,100.00%,94.44%
無訊號區間長度中位數,20.0,—,20.0
無訊號區間長度最長,20.0,—,20.0
有訊號區間長度中位數,61.0,59.0,63.0
等待日數 最短,2.0,2.0,3.0
等待日數 中位數,12.0,12.0,12.0


In [14]:
g_all = compare[has_sig]
record("訊號統計", True,
       f"全期間 {len(compare)} 區間／有訊號 {int(has_sig.sum())}；"
       f"等待日數 {g_all['等待交易日數'].min():.0f}~{g_all['等待交易日數'].max():.0f}"
       f"（中位數 {g_all['等待交易日數'].median():.0f}）；"
       f"進場價差異 {g_all['進場價差異%'].min():+.2%}~{g_all['進場價差異%'].max():+.2%}"
       f"（中位數 {g_all['進場價差異%'].median():+.2%}），"
       f"B 組買得更便宜的比例 {(g_all['進場價差異%'] < 0).mean():.1%}")

===> [通過] 訊號統計
      全期間 50 區間／有訊號 49；等待日數 2~36（中位數 12）；進場價差異 -6.17%~+19.57%（中位數 +1.38%），B 組買得更便宜的比例 36.7%


---
## 7. 未產生訊號的區間分析

單獨列出所有未產生訊號的區間，逐段回答「為什麼沒有訊號」。

**這一格的目的是確認「沒訊號」是規則的自然結果，不是程式 bug。**
可能的原因只有三種：

1. 區間內從未出現 `RSI3 < 30` —— 一路上漲沒回檔，旗標從未啟動
2. 旗標啟動了，但到區間結束都沒等到 `RSI3` 上穿 `RSI5`
3. 訊號成立了，但次日已轉 `FLAT`，依規則作廢

若出現第四種無法解釋的情況，就是實作有問題。

In [15]:
rows = []
for b in compare.index[~has_sig]:
    seg = df[df["regime_block"] == b]
    r3 = seg["RSI3"].to_numpy(); r5 = seg["RSI5"].to_numpy()
    ov = np.where(r3 < RSI_OVERSOLD)[0]

    if compare.loc[b, "是否產生訊號"] == "★ 訊號作廢":
        reason = "訊號成立但次日已轉 FLAT，依規則作廢"
    elif len(ov) == 0:
        reason = f"區間內從未出現 RSI3 < {RSI_OVERSOLD}（旗標未曾啟動）"
    else:
        first = ov[0]
        cross = np.where((r3[1:] > r5[1:]) & (r3[:-1] <= r5[:-1]))[0] + 1
        after = cross[cross >= first]
        reason = ("旗標啟動後至區間結束未出現 RSI3 上穿 RSI5"
                  if len(after) == 0 else "★ 無法解釋 —— 需人工檢查")

    rows.append({
        "區間#": b,
        "區間起日": compare.loc[b, "區間起日"],
        "區間結束日": compare.loc[b, "區間結束日"],
        "區間長度": compare.loc[b, "區間長度"],
        f"RSI3 < {RSI_OVERSOLD} 天數": int(len(ov)),
        f"首次 RSI3 < {RSI_OVERSOLD}": f"{seg.index[ov[0]]:%Y-%m-%d}" if len(ov) else "—",
        "區間內最低 RSI3": seg["RSI3"].min(),
        "未產生訊號的原因": reason,
    })

NO_SIG_COLS = ["區間#", "區間起日", "區間結束日", "區間長度",
               f"RSI3 < {RSI_OVERSOLD} 天數", f"首次 RSI3 < {RSI_OVERSOLD}",
               "區間內最低 RSI3", "未產生訊號的原因"]
no_sig = pd.DataFrame(rows, columns=NO_SIG_COLS).set_index("區間#")

if no_sig.empty:
    print(f"研究期間全部 {len(compare)} 個 {LONG_OK} 區間都產生了進場訊號，"
          "沒有需要解釋的無訊號區間。")
else:
    display(no_sig.style.format({"區間內最低 RSI3": "{:.2f}"}))

,區間起日,區間結束日,區間長度,RSI3 < 30 天數,首次 RSI3 < 30,區間內最低 RSI3,未產生訊號的原因
區間#,,,,,,,
34,2018-06-01,2018-06-29,20,8,2018-06-14,4.94,訊號成立但次日已轉 FLAT，依規則作廢


In [16]:
if no_sig.empty:
    unexplained = 0
    detail = (f"全部 {len(compare)} 個 {LONG_OK} 區間都產生了進場訊號，"
              "無訊號區間數為 0，因此沒有需要解釋的情況")
else:
    unexplained = int(no_sig["未產生訊號的原因"].str.contains("無法解釋").sum())
    reason_cnt = no_sig["未產生訊號的原因"].value_counts().to_frame("區間數")
    reason_cnt.index.name = "原因"
    display(reason_cnt)
    detail = (f"{len(no_sig)} 個無訊號區間全部可歸因於規則本身（長度中位數 "
              f"{no_sig['區間長度'].median():.0f} 交易日、"
              f"最長 {no_sig['區間長度'].max():.0f}），無無法解釋的情況")

record("未產生訊號的區間可完全解釋", unexplained == 0, detail
       if unexplained == 0 else f"有 {unexplained} 個區間無法解釋，需人工檢查")

,區間數
原因,
訊號成立但次日已轉 FLAT，依規則作廢,1


===> [通過] 未產生訊號的區間可完全解釋
      1 個無訊號區間全部可歸因於規則本身（長度中位數 20 交易日、最長 20），無無法解釋的情況


---
## 8. 視覺化

### 圖 1｜全期間訊號分布

收盤價對數座標，灰色背景為 `FLAT` 區間，
橙色三角為 C 組進場日、藍色圓點為 B 組進場日，虛線為 IS/OOS 分界。

In [17]:
def shade_flat(ax, frame):
    s = (frame["regime"] == LONG_OK).astype(int)
    gid = (s != s.shift(1)).cumsum()
    for _, seg in s.groupby(gid):
        if seg.iloc[0] == 0:
            i = frame.index.get_loc(seg.index[-1])
            end = frame.index[min(i + 1, len(frame) - 1)]
            ax.axvspan(seg.index[0], end, color="grey", alpha=0.20, lw=0)

c_days = pd.to_datetime(compare["C組進場日"])
b_days = pd.to_datetime(compare.loc[has_sig, "B組進場日"])

fig, ax = plt.subplots(figsize=(15, 7))
ax.plot(df.index, df["Close"], lw=0.9, color="#3b6ea5", label="Close")
shade_flat(ax, df)
ax.scatter(c_days, df.loc[c_days, "Open"], marker="^", s=70, color="#e08a3c",
           zorder=6, label="C entry (block start)")
ax.scatter(b_days, df.loc[b_days, "Open"], marker="o", s=45, color="#b3261e",
           zorder=7, label="B entry (RSI signal + 1)")
ax.axvline(pd.Timestamp(OOS_START), color="black", ls="--", lw=1.2)
ax.text(pd.Timestamp(OOS_START), df["Close"].max(), "  IS | OOS",
        va="top", ha="left", fontsize=10)
ax.set_yscale("log")
ax.set_title("Entry timing: C (filter only) vs B (filter + RSI) — shaded = FLAT")
ax.set_xlabel("Date"); ax.set_ylabel("Close (log)")
ax.legend(loc="upper left"); ax.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

C:\Users\king5\AppData\Local\Temp\ipykernel_14576\559804305.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 圖 2–5｜四個代表性區間

挑選規則寫死如下，不做人為選圖：

1. B 組進場價**最便宜**的區間（`進場價差異%` 最小）
2. B 組進場價**最貴**的區間（`進場價差異%` 最大）
3. **未產生訊號**的區間中最長的一段
4. **等待天數最長**的區間

每張三個 panel：上為收盤價與進場點，中為 RSI3／RSI5 與 30 水平線，
下為 `pullback_flag` 的布林序列。

In [18]:
picks = [
    (int(compare.loc[has_sig, "進場價差異%"].idxmin()), "Cheapest B entry"),
    (int(compare.loc[has_sig, "進場價差異%"].idxmax()), "Most expensive B entry"),
]
# 第 3 張原訂為「未產生訊號的最長區間」；本次執行沒有無訊號區間，
# 改用最長的 LONG_OK 區間，並在標題註明替代原因。
if not no_sig.empty:
    picks.append((int(no_sig["區間長度"].idxmax()), "Longest block with NO signal"))
else:
    picks.append((int(compare["區間長度"].idxmax()),
                  "Longest LONG_OK block (no signal-less block exists)"))
picks.append((int(compare.loc[has_sig, "等待交易日數"].idxmax()), "Longest wait"))

def plot_block(b, title):
    seg = df[df["regime_block"] == b]
    # 前後各留 10 個交易日的脈絡
    i0 = max(df.index.get_loc(seg.index[0]) - 10, 0)
    i1 = min(df.index.get_loc(seg.index[-1]) + 10, len(df) - 1)
    ctx = df.iloc[i0: i1 + 1]

    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True,
                             gridspec_kw={"height_ratios": [3, 2, 1]})
    axes[0].plot(ctx.index, ctx["Close"], lw=1.1, color="#3b6ea5", label="Close")
    shade_flat(axes[0], ctx)
    cd = pd.Timestamp(compare.loc[b, "C組進場日"])
    axes[0].scatter([cd], [df.loc[cd, "Open"]], marker="^", s=110,
                    color="#e08a3c", zorder=6, label="C entry")
    if compare.loc[b, "B組進場日"] != "—":
        bd = pd.Timestamp(compare.loc[b, "B組進場日"])
        axes[0].scatter([bd], [df.loc[bd, "Open"]], marker="o", s=80,
                        color="#b3261e", zorder=7, label="B entry")
        axes[0].axvline(bd, color="#b3261e", ls="--", lw=1.0)
    axes[0].set_title(f"Block #{b} — {title}  "
                      f"({compare.loc[b, '區間起日']} to {compare.loc[b, '區間結束日']})")
    axes[0].set_ylabel("Close"); axes[0].legend(loc="best"); axes[0].grid(alpha=0.3)

    axes[1].plot(ctx.index, ctx["RSI3"], lw=1.0, color="#3b6ea5", label="RSI(3)")
    axes[1].plot(ctx.index, ctx["RSI5"], lw=1.0, color="#e08a3c", label="RSI(5)")
    axes[1].axhline(RSI_OVERSOLD, color="grey", ls="--", lw=0.9)
    sg = seg.index[seg["entry_signal"]]
    if len(sg):
        axes[1].scatter(sg, df.loc[sg, "RSI3"], marker="*", s=200,
                        color="#b3261e", zorder=7, label="Signal day")
    ov = seg.index[seg["RSI3"] < RSI_OVERSOLD]
    if len(ov):
        axes[1].scatter(ov, df.loc[ov, "RSI3"], marker="v", s=30,
                        color="#2e7d32", zorder=6, label=f"RSI3 < {RSI_OVERSOLD}")
    axes[1].set_ylim(0, 100); axes[1].set_ylabel("RSI")
    axes[1].legend(loc="best", ncol=2); axes[1].grid(alpha=0.3)

    axes[2].fill_between(ctx.index, ctx["pullback_flag"].astype(int),
                         color="#6a4c93", alpha=0.7, step="post")
    axes[2].set_ylim(-0.1, 1.1); axes[2].set_yticks([0, 1])
    axes[2].set_ylabel("pullback\nflag"); axes[2].set_xlabel("Date")
    axes[2].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

for b, t in picks:
    plot_block(b, t)

C:\Users\king5\AppData\Local\Temp\ipykernel_14576\968100118.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()
C:\Users\king5\AppData\Local\Temp\ipykernel_14576\968100118.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


C:\Users\king5\AppData\Local\Temp\ipykernel_14576\968100118.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


C:\Users\king5\AppData\Local\Temp\ipykernel_14576\968100118.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 圖 6｜進場價差異分布

所有有訊號區間的「進場價差異%」，依時間排序。
綠色（負值）代表 B 組買得比 C 組便宜，紅色（正值）代表買得更貴。

In [19]:
g = compare[has_sig].copy()
g["x"] = pd.to_datetime(g["區間起日"])
g = g.sort_values("x")

fig, ax = plt.subplots(figsize=(15, 6))
colors = np.where(g["進場價差異%"] < 0, "#2e7d32", "#b3261e")
ax.bar(range(len(g)), g["進場價差異%"] * 100, color=colors)
ax.axhline(0, color="black", lw=0.8)
ax.axhline(g["進場價差異%"].median() * 100, color="#3b6ea5", ls="--", lw=1.2,
           label=f"median {g['進場價差異%'].median():+.2%}")
ax.set_xticks(range(len(g)))
ax.set_xticklabels([f"#{i}\n{d:%Y-%m}" for i, d in zip(g.index, g["x"])],
                   fontsize=8, rotation=45)
ax.set_title("B entry price vs C entry price (negative = B bought cheaper)")
ax.set_xlabel("LONG_OK block"); ax.set_ylabel("Difference (%)")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

C:\Users\king5\AppData\Local\Temp\ipykernel_14576\3567472914.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
## 9. 存檔

`data/signals.csv` —— 沿用 `regime.csv` 全部欄位，加上
`pullback_flag`、`entry_signal`、`b_entry_day`、`regime_block`。
另加 `signal_void` 一欄，標記依規則作廢的訊號（不記錄的話下游看不出差別）。

**保留 1999 年資料不裁切**，該期間新增欄位為 False / 空值。

`data/signal_comparison.csv` —— 區塊 5 的 C／B 對照表。

In [20]:
NEW_COLS = ["regime_block", "pullback_flag", "entry_signal", "signal_void", "b_entry_day"]

sig_out = raw.copy()
for c in NEW_COLS:
    if c == "regime_block":
        sig_out[c] = np.nan
    else:
        sig_out[c] = False
sig_out.loc[df.index, NEW_COLS] = df[NEW_COLS].values

sig_out.to_csv(DATA_OUT, date_format="%Y-%m-%d")
compare.to_csv(COMPARE_OUT, encoding="utf-8-sig")

back = pd.read_csv(DATA_OUT, index_col="Date", parse_dates=True)
back_cmp = pd.read_csv(COMPARE_OUT, index_col="區間#")

num_cols = ["Open", "High", "Low", "Close", "Volume", "MA", "RSI3", "RSI5"]
ok = (
    len(back) == len(sig_out)
    and list(back.columns) == list(sig_out.columns)
    and np.allclose(back[num_cols].to_numpy(), sig_out[num_cols].to_numpy(), equal_nan=True)
    and int(back["entry_signal"].sum()) == int(sig_out["entry_signal"].sum())
    and int(back["b_entry_day"].sum()) == int(sig_out["b_entry_day"].sum())
    and int(back["pullback_flag"].sum()) == int(sig_out["pullback_flag"].sum())
    and len(back_cmp) == len(compare)
)

info = pd.DataFrame({"值": [
    DATA_OUT, f"{os.path.getsize(DATA_OUT):,} bytes", f"{len(back):,}",
    ", ".join(back.columns),
    f"{back.index.min():%Y-%m-%d} ~ {back.index.max():%Y-%m-%d}",
    f"{int((back.index < pd.Timestamp(STUDY_START)).sum()):,}",
    f"{int(back['entry_signal'].sum())}",
    f"{int(back['b_entry_day'].sum())}",
    COMPARE_OUT, f"{len(back_cmp)}",
]}, index=["訊號檔", "  └ 大小", "  └ 筆數", "  └ 欄位", "  └ 期間",
           "  └ 1999 暖身期筆數（已保留）", "  └ entry_signal 天數",
           "  └ b_entry_day 天數", "對照表", "  └ 區間數"])
info.index.name = "項目"
display(info)

record("存檔與讀回一致性", ok,
       f"{DATA_OUT}（{len(back):,} 筆 × {len(back.columns)} 欄）與 "
       f"{COMPARE_OUT}（{len(back_cmp)} 區間）皆已寫入，讀回完全一致"
       if ok else "讀回的資料與記憶體中不一致，請停止並人工確認")

,值
項目,
訊號檔,data/signals.csv
└ 大小,"1,192,208 bytes"
└ 筆數,"6,632"
└ 欄位,"Open, High, Low, Close, Volume, MA, RSI3, RSI5..."
└ 期間,1999-01-05 ~ 2026-01-20
└ 1999 暖身期筆數（已保留）,241
└ entry_signal 天數,50
└ b_entry_day 天數,49
對照表,data/signal_comparison.csv


===> [通過] 存檔與讀回一致性
      data/signals.csv（6,632 筆 × 16 欄）與 data/signal_comparison.csv（50 區間）皆已寫入，讀回完全一致


---
## 10. 小結

In [21]:
summary = pd.DataFrame(CHECKS)
display(summary)

n_fail = int((summary["結果"] == "異常").sum())
print(f"\n共 {len(summary)} 項驗證，通過 {len(summary) - n_fail} 項，異常 {n_fail} 項")

,驗證項目,結果,說明
0,資料載入一致性,通過,"6,632 筆與階段 3 完全一致；研究期間切出 6,391 個交易日（2000-01-04..."
1,LONG_OK 區間切分,通過,研究期間切出 50 個 LONG_OK 區間，與階段 4 的 C 組交易筆數 50 一致；最...
2,(a) 旗標邏輯,通過,161 個 pullback_flag 為 True 的交易日全部可回溯到同一區間內的 RS...
3,(b) 訊號條件,通過,50 個 entry_signal 全部同時滿足四個條件：regime 為 LONG_OK、...
4,(c) 每區間最多一次訊號,通過,50 個 LONG_OK 區間中，50 個產生 1 次訊號、0 個未產生訊號，無任何區間超過...
5,(d) 時序：訊號日 t，進場日 t+1,通過,49 個有效訊號的進場日皆為訊號日的下一個交易日（間隔恰為 1），進場日 regime 皆為...
6,訊號清單,通過,50 個 LONG_OK 區間中，49 個產生有效訊號、1 個未產生（含訊號作廢者 1 個）...
7,訊號統計,通過,全期間 50 區間／有訊號 49；等待日數 2~36（中位數 12）；進場價差異 -6.17...
8,未產生訊號的區間可完全解釋,通過,1 個無訊號區間全部可歸因於規則本身（長度中位數 20 交易日、最長 20），無無法解釋的情況
9,存檔與讀回一致性,通過,"data/signals.csv（6,632 筆 × 16 欄）與 data/signal_..."



共 10 項驗證，通過 10 項，異常 0 項


In [22]:
key = pd.DataFrame({"值": [
    f"{len(compare)}",
    f"{int(has_sig.sum())}（{has_sig.mean():.1%}）",
    f"{int((~has_sig).sum())}（{(~has_sig).mean():.1%}）",
    f"{int((compare['是否產生訊號'] == '★ 訊號作廢').sum())}",
    "—（無訊號區間數為 0）" if no_sig.empty else
    f"{no_sig['區間長度'].median():.0f} / {no_sig['區間長度'].max():.0f}",
    f"{g_all['區間長度'].median():.0f}",
    f"{g_all['等待交易日數'].min():.0f} / {g_all['等待交易日數'].median():.0f} / "
    f"{g_all['等待交易日數'].mean():.1f} / {g_all['等待交易日數'].max():.0f}",
    f"{g_all['進場價差異%'].median():+.2%}",
    f"{g_all['進場價差異%'].mean():+.2%}",
    f"{g_all['進場價差異%'].min():+.2%} / {g_all['進場價差異%'].max():+.2%}",
    f"{(g_all['進場價差異%'] < 0).mean():.1%}",
]}, index=[
    f"{LONG_OK} 區間總數", "有訊號區間數", "無訊號區間數", "  └ 其中訊號作廢",
    "無訊號區間長度 中位數/最長", "有訊號區間長度 中位數",
    "等待交易日數 最短/中位數/平均/最長",
    "進場價差異 中位數", "進場價差異 平均",
    "進場價差異 最好/最差", "B 組買得更便宜的比例"])
key.index.name = "關鍵數字"
display(key)

,值
關鍵數字,
LONG_OK 區間總數,50
有訊號區間數,49（98.0%）
無訊號區間數,1（2.0%）
└ 其中訊號作廢,1
無訊號區間長度 中位數/最長,20 / 20
有訊號區間長度 中位數,61
等待交易日數 最短/中位數/平均/最長,2 / 12 / 13.0 / 36
進場價差異 中位數,+1.38%
進場價差異 平均,+1.81%


### 本階段結論

以下只陳述數字與觀察到的事實，
不評價 RSI 有沒有用、不預測 B 組回測結果、不建議調整參數或規則。

**狀態機實作已驗證**

- **旗標邏輯**：每個 `pullback_flag == True` 的交易日都能回溯到同一區間內的
  `RSI3 < 30`，且中間沒有未清除的訊號；`FLAT` 期間旗標全為 `False`。
- **訊號條件**：每一個 `entry_signal` 都同時滿足四個條件
  （`regime == LONG_OK`、當日 `RSI3 > RSI5`、前一日 `RSI3 <= RSI5`、
  區間內先前出現過 `RSI3 < 30`、區間內此前未進場），以程式化斷言逐筆確認。
- **每區間最多一次**：無任何 `LONG_OK` 區間產生超過一次訊號。
- **時序**：所有有效訊號的進場日恰為訊號日的**下一個交易日**，
  進場日 `regime` 皆為 `LONG_OK`，訊號日與進場日不重疊。

**兩層的 shift 方向都正確**：第一層 `regime` 未再 shift（階段 3 已處理），
第二層 RSI 訊號 shift 一天後才成交。這兩者容易互相污染，已分別確認。

**訊號數量與時點**

`LONG_OK` 區間總數、有訊號與無訊號的區間數與比例、
無訊號區間的長度分布，見上方「關鍵數字」表與區塊 6（含 IS／OOS 分組）。

**未產生訊號的區間全部可解釋**

區塊 7 逐段列出原因，全部歸因於規則本身
（區間內未出現 `RSI3 < 30`、旗標啟動後區間就結束、或訊號次日轉 `FLAT` 而作廢），
**沒有無法解釋的情況**，確認「沒訊號」不是程式 bug。

**等待天數與進場價差異**

等待交易日數的最短／中位數／平均／最長，
以及進場價差異的中位數、平均、最好、最差與「B 組買得更便宜的比例」，
見「關鍵數字」表；IS／OOS 分組見區塊 6，逐區間的長條圖見區塊 8 圖 6。

---

### 產出

- `data/signals.csv` —— 原欄位 + `regime_block` / `pullback_flag` / `entry_signal` /
  `signal_void` / `b_entry_day`，保留 1999 暖身期
- `data/signal_comparison.csv` —— 每個 `LONG_OK` 區間的 C／B 進場對照

### 下一階段

B 組回測與 A／B／C 三組對照屬於階段 6。
請先人工確認上述訊號清單與未產生訊號的區間分析，再進行下一階段。